# Baskonia, un gigante medible

## Veinticinco años de Euroliga frente al pesimismo de un 0-2

**autor:** Roberto Cantero  
**proyecto:** Nébula  
**corte editorial de la temporada actual:** 1 de octubre de 2026

Este notebook no parte de la conclusión de que Baskonia sea un gigante. La somete a dos pruebas:

1. **Huella histórica:** permanencia, volumen de victorias, rendimiento relativo y presencia en las grandes rondas.
2. **Valor de un mal comienzo:** cuánto anticipan realmente las primeras 1, 2, 3, 5 y 10 jornadas del resultado final.


## Criterio editorial

La grandeza no se reducirá a un índice opaco. Se mostrarán por separado cuatro dimensiones: **longevidad, victorias, eficiencia y profundidad competitiva**. Baskonia no ha ganado la Euroliga; ocultarlo debilitaría la pieza. La pregunta interesante es si su continuidad cerca de la élite constituye una excepción histórica.

La segunda parte tampoco intentará adivinar el futuro con una muestra mínima. Medirá cómo aumenta la señal estadística a medida que avanza la temporada y distinguirá entre una observación, una asociación y una predicción.


## Fuentes y alcance

- Partidos y resultados: API pública utilizada por EuroLeague Basketball.
- Grandes rondas de Baskonia: perfil histórico y Media Centre de EuroLeague.
- Historia moderna: temporadas terminadas entre 2000-01 y 2025-26, según disponibilidad del servicio.
- Análisis de comienzos: 2016-17 a 2025-26, cuando la competición adopta una liga regular comparable.
- La campaña 2026-27 se conserva como una fotografía separada; nunca entra en el entrenamiento histórico.

La API es pública, pero no está documentada como un producto estable. Por eso se guardan las respuestas originales, se comprueba el esquema mínimo y se enumeran los fallos de descarga.


## Preparación del análisis

In [ ]:
from pathlib import Path
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='whitegrid', context='notebook')

AZUL_BASKONIA = '#003B7A'
ROJO_BASKONIA = '#E30613'
GRIS = '#9AA0A6'

PROJECT_DIR = Path.cwd()
RAW_DIR = PROJECT_DIR / 'data' / 'raw'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
EXPORT_DIR = PROJECT_DIR / 'exports'

for directory in (RAW_DIR, PROCESSED_DIR, EXPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print('El notebook creará sus carpetas de datos junto al archivo descargado')

BASKONIA = 'BAS'
COMPLETED_START_YEARS = list(range(2000, 2026))
LEAGUE_ERA_START = 2016
CURRENT_START_YEAR = 2026
AS_OF_DATE = pd.Timestamp('2026-10-01 23:59:59', tz='UTC')

print(f'Proyecto: {PROJECT_DIR}')
print(f'Temporadas terminadas solicitadas: {len(COMPLETED_START_YEARS)}')


## Descargar y congelar los resultados

Cada temporada requiere una sola consulta. La respuesta se guarda en data/raw y las ejecuciones posteriores trabajan contra esa copia. Se introduce una pausa respetuosa entre peticiones nuevas para no sobrecargar el servicio.

In [ ]:
BASE_URL = 'https://api-live.euroleague.net/v2/competitions/E/seasons'

session = requests.Session()
session.headers.update({
    'Accept': 'application/json',
    'User-Agent': 'Nebula-Baskonia-Historia/1.0 (research notebook)'
})

def fetch_season_games(start_year, refresh=False, pause_seconds=6.5):
    season_code = f'E{start_year}'
    cache_file = RAW_DIR / f'{season_code}_games.json'

    if cache_file.exists() and not refresh:
        with cache_file.open('r', encoding='utf-8') as file:
            return json.load(file), 'cache'

    url = f'{BASE_URL}/{season_code}/games'
    response = session.get(url, params={'limit': 1000}, timeout=45)
    response.raise_for_status()
    payload = response.json()

    with cache_file.open('w', encoding='utf-8') as file:
        json.dump(payload, file, ensure_ascii=False, indent=2)

    time.sleep(pause_seconds)
    return payload, 'api'

def records_from_payload(payload):
    if isinstance(payload, dict) and isinstance(payload.get('data'), list):
        return payload['data']
    if isinstance(payload, list):
        return payload
    raise ValueError('La respuesta no contiene una lista de partidos reconocible')


In [ ]:
payloads = {}
download_log = []

for start_year in COMPLETED_START_YEARS:
    try:
        payload, origin = fetch_season_games(start_year)
        games_count = len(records_from_payload(payload))
        payloads[start_year] = payload
        download_log.append({
            'season': f'{start_year}-{str(start_year + 1)[-2:]}',
            'start_year': start_year,
            'status': 'ok',
            'origin': origin,
            'records': games_count,
            'error': None
        })
        print(f'{start_year}: {games_count} partidos ({origin})')
    except Exception as exc:
        download_log.append({
            'season': f'{start_year}-{str(start_year + 1)[-2:]}',
            'start_year': start_year,
            'status': 'error',
            'origin': None,
            'records': 0,
            'error': str(exc)
        })
        print(f'{start_year}: ERROR -> {exc}')

df_download_log = pd.DataFrame(download_log)
display(df_download_log)


### Validar la cobertura antes de analizar

El notebook no rellena huecos silenciosamente. Si una temporada antigua no está disponible, se conserva el fallo en la tabla y el periodo efectivo se declara en los resultados.

In [ ]:
available_years = sorted(payloads)
failed_years = df_download_log.loc[df_download_log['status'].eq('error'), 'start_year'].tolist()

assert available_years, 'No se descargó ninguna temporada'
assert 2016 in available_years, 'Falta 2016-17, imprescindible para el análisis de comienzos'

print(f'Primera temporada disponible: {min(available_years)}-{str(min(available_years)+1)[-2:]}')
print(f'Última temporada terminada: {max(available_years)}-{str(max(available_years)+1)[-2:]}')
print(f'Temporadas disponibles: {len(available_years)}')
print(f'Temporadas no disponibles: {failed_years or "ninguna"}')


## Normalizar el esquema de partidos

La API anida club y marcador dentro de local y road. Estas funciones conservan únicamente las variables necesarias y toleran pequeñas variaciones de nombres entre temporadas.

In [ ]:
def first_value(mapping, keys, default=None):
    if not isinstance(mapping, dict):
        return default
    for key in keys:
        value = mapping.get(key)
        if value is not None:
            return value
    return default

def nested_label(value):
    if isinstance(value, dict):
        code = first_value(value, ['code', 'phaseTypeCode', 'abbreviatedName'], '')
        name = first_value(value, ['name', 'alias', 'description'], '')
        return ' | '.join(str(item) for item in (code, name) if item)
    return '' if value is None else str(value)

def club_identity(side):
    club = side.get('club', {}) if isinstance(side, dict) else {}
    code = first_value(club, ['code', 'clubCode', 'abbreviatedName'])
    name = first_value(club, ['name', 'editorialName', 'abbreviatedName'], code)
    return code, name

def score_value(side):
    score = side.get('score') if isinstance(side, dict) else None
    if isinstance(score, dict):
        score = first_value(score, ['total', 'score', 'points'])
    return pd.to_numeric(score, errors='coerce')

def flatten_game(game, start_year):
    local = game.get('local') or {}
    road = game.get('road') or {}
    local_code, local_name = club_identity(local)
    road_code, road_name = club_identity(road)
    phase_text = nested_label(game.get('phaseType'))
    group_text = nested_label(game.get('group'))

    return {
        'season_start': start_year,
        'season': f'{start_year}-{str(start_year + 1)[-2:]}',
        'game_code': first_value(game, ['gameCode', 'code']),
        'identifier': first_value(game, ['identifier', 'id']),
        'date': first_value(game, ['date', 'localDate']),
        'played': game.get('played') is True or str(game.get('played')).lower() in {'true', '1'},
        'game_status': nested_label(game.get('gameStatus')),
        'phase': phase_text,
        'group': group_text,
        'local_code': local_code,
        'local_name': local_name,
        'local_score': score_value(local),
        'road_code': road_code,
        'road_name': road_name,
        'road_score': score_value(road)
    }

rows = []
for start_year, payload in payloads.items():
    rows.extend(flatten_game(game, start_year) for game in records_from_payload(payload))

games = pd.DataFrame(rows)
games['date'] = pd.to_datetime(games['date'], errors='coerce', utc=True)
games['game_code'] = pd.to_numeric(games['game_code'], errors='coerce')

played_games = games.loc[
    games['played']
    & games['local_score'].notna()
    & games['road_score'].notna()
    & games['local_code'].notna()
    & games['road_code'].notna()
].copy()

assert not played_games.empty
assert played_games[['season', 'game_code']].duplicated().sum() == 0
assert (played_games['local_score'] != played_games['road_score']).all(), 'Un partido de baloncesto no puede acabar empatado'

print('Partidos recibidos:', len(games))
print('Partidos terminados y válidos:', len(played_games))
display(played_games.head())


### Pasar de partido a equipo-partido

Cada encuentro se transforma en dos observaciones: una desde la perspectiva del local y otra desde la del visitante. Esto permite agregar temporadas, comienzos y trayectorias sin duplicar lógica.

In [ ]:
common = ['season_start', 'season', 'game_code', 'identifier', 'date', 'phase', 'group']

home = played_games[common + ['local_code', 'local_name', 'local_score', 'road_code', 'road_name', 'road_score']].copy()
home.columns = common + ['team_code', 'team_name', 'points_for', 'opponent_code', 'opponent_name', 'points_against']
home['is_home'] = True

away = played_games[common + ['road_code', 'road_name', 'road_score', 'local_code', 'local_name', 'local_score']].copy()
away.columns = common + ['team_code', 'team_name', 'points_for', 'opponent_code', 'opponent_name', 'points_against']
away['is_home'] = False

team_games = pd.concat([home, away], ignore_index=True)
team_games['win'] = (team_games['points_for'] > team_games['points_against']).astype(int)
team_games['point_diff'] = team_games['points_for'] - team_games['points_against']
team_games['phase_upper'] = team_games['phase'].fillna('').str.upper()
team_games['is_regular_season'] = team_games['phase_upper'].str.contains(r'(^|\|\s*)RS($|\s*\|)|REGULAR', regex=True)

assert len(team_games) == 2 * len(played_games)
display(team_games.head())


## 1. La huella histórica de Baskonia

Primero se comparan todos los partidos disputados en la competición. Para reducir el sesgo de clubes con una aparición breve, la tabla principal exige al menos diez temporadas.

In [ ]:
club_history = (
    team_games
    .groupby('team_code', as_index=False)
    .agg(
        club=('team_name', 'last'),
        seasons=('season', 'nunique'),
        games=('win', 'size'),
        wins=('win', 'sum'),
        point_diff=('point_diff', 'sum')
    )
)
club_history['win_pct'] = 100 * club_history['wins'] / club_history['games']
club_history['point_diff_per_game'] = club_history['point_diff'] / club_history['games']

established_clubs = (
    club_history
    .query('seasons >= 10')
    .sort_values(['wins', 'win_pct'], ascending=False)
    .reset_index(drop=True)
)
established_clubs.index = established_clubs.index + 1
established_clubs.index.name = 'rank_wins'

display(established_clubs.round({'win_pct': 1, 'point_diff_per_game': 2}))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

colors = np.where(established_clubs['team_code'].eq(BASKONIA), ROJO_BASKONIA, AZUL_BASKONIA)
sizes = 35 + established_clubs['win_pct'] ** 1.35

ax.scatter(
    established_clubs['seasons'],
    established_clubs['wins'],
    s=sizes,
    c=colors,
    alpha=0.82,
    edgecolor='white',
    linewidth=1
)

for _, row in established_clubs.iterrows():
    if row['team_code'] == BASKONIA or row['wins'] >= established_clubs['wins'].quantile(0.70):
        ax.annotate(
            row['team_code'],
            (row['seasons'], row['wins']),
            xytext=(5, 5),
            textcoords='offset points',
            fontsize=9,
            fontweight='bold' if row['team_code'] == BASKONIA else 'normal'
        )

ax.set(
    title='Permanecer también es competir',
    xlabel='Temporadas disputadas en el periodo disponible',
    ylabel='Victorias totales'
)
ax.text(0, 1.02, 'El tamaño representa el porcentaje de victorias; solo clubes con 10+ temporadas', transform=ax.transAxes, color='#555555')
sns.despine()
plt.tight_layout()
plt.show()


### Una final y cinco Final Four

La Euroliga contabiliza seis apariciones de Baskonia en la gran cita de semifinales desde 1988. El matiz importa: en 2001 no se disputó una Final Four, sino eliminatorias y una final al mejor de cinco. Por eso la formulación editorial rigurosa es **una final en 2001 y cinco Final Four posteriores**.

In [ ]:
great_rounds = pd.DataFrame([
    {'season': '2000-01', 'stage': 'Final', 'result': 'Subcampeon', 'format': 'Final al mejor de cinco'},
    {'season': '2004-05', 'stage': 'Final Four', 'result': 'Subcampeon', 'format': 'Final Four'},
    {'season': '2005-06', 'stage': 'Final Four', 'result': 'Semifinalista', 'format': 'Final Four'},
    {'season': '2006-07', 'stage': 'Final Four', 'result': 'Semifinalista', 'format': 'Final Four'},
    {'season': '2007-08', 'stage': 'Final Four', 'result': 'Semifinalista', 'format': 'Final Four'},
    {'season': '2015-16', 'stage': 'Final Four', 'result': 'Semifinalista', 'format': 'Final Four'},
])

assert len(great_rounds) == 6
assert great_rounds['season'].nunique() == 6

display(great_rounds[['season', 'stage', 'result', 'format']])
print('Finales:', great_rounds['result'].eq('Subcampeon').sum())
print('Final Four:', great_rounds['stage'].eq('Final Four').sum())


### Baskonia temporada a temporada

El porcentaje se calcula sobre todos los partidos europeos de cada campaña. No convierte formatos distintos en idénticos, pero permite observar ciclos sin borrar Top 16, playoffs o Final Four.

In [ ]:
team_seasons = (
    team_games
    .groupby(['season_start', 'season', 'team_code'], as_index=False)
    .agg(
        team_name=('team_name', 'last'),
        games=('win', 'size'),
        wins=('win', 'sum'),
        point_diff=('point_diff', 'sum')
    )
)
team_seasons['win_pct'] = 100 * team_seasons['wins'] / team_seasons['games']
team_seasons['point_diff_per_game'] = team_seasons['point_diff'] / team_seasons['games']
team_seasons['win_pct_rank'] = team_seasons.groupby('season')['win_pct'].rank(method='average', ascending=False)
team_seasons['teams_in_season'] = team_seasons.groupby('season')['team_code'].transform('nunique')
team_seasons['relative_percentile'] = 100 * (
    1 - (team_seasons['win_pct_rank'] - 1) / (team_seasons['teams_in_season'] - 1).clip(lower=1)
)

baskonia_seasons = team_seasons.loc[team_seasons['team_code'].eq(BASKONIA)].copy()
baskonia_seasons = baskonia_seasons.sort_values('season_start')

display(baskonia_seasons.round({'win_pct': 1, 'point_diff_per_game': 2, 'relative_percentile': 1}))


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    baskonia_seasons['season'],
    baskonia_seasons['win_pct'],
    color=AZUL_BASKONIA,
    linewidth=2.4,
    marker='o',
    markersize=5
)
ax.axhline(50, color=GRIS, linestyle='--', linewidth=1.2, label='50% de victorias')

deep_seasons = set(great_rounds['season'])
deep = baskonia_seasons.loc[baskonia_seasons['season'].isin(deep_seasons)]
ax.scatter(deep['season'], deep['win_pct'], color=ROJO_BASKONIA, s=75, zorder=3, label='Final / Final Four')

ax.set(
    title='Baskonia no fue un visitante ocasional de la élite',
    xlabel='Temporada',
    ylabel='Porcentaje de victorias en todos los partidos'
)
ax.tick_params(axis='x', rotation=60)
ax.legend(frameon=False)
sns.despine()
plt.tight_layout()
plt.show()


## 2. ¿Cuándo empieza a importar un mal comienzo?

Para esta pregunta se utiliza solo la liga regular desde 2016-17. En cada equipo-temporada se compara el balance acumulado después de k partidos con el porcentaje final de victorias.

La posición Top 8 se aproxima ordenando victorias y diferencia de puntos. No sustituye los desempates oficiales; se utiliza únicamente como indicador exploratorio y se etiqueta como proxy.

In [ ]:
league_games = team_games.loc[
    team_games['season_start'].between(LEAGUE_ERA_START, 2025)
    & team_games['is_regular_season']
].copy()

if league_games.empty:
    raise ValueError('No se reconoció la fase regular. Revisa los valores únicos de phase.')

league_games = league_games.sort_values(['season_start', 'team_code', 'date', 'game_code'])
league_games['game_number'] = league_games.groupby(['season_start', 'team_code']).cumcount() + 1

season_finals = (
    league_games
    .groupby(['season_start', 'season', 'team_code'], as_index=False)
    .agg(
        team_name=('team_name', 'last'),
        final_games=('win', 'size'),
        final_wins=('win', 'sum'),
        final_point_diff=('point_diff', 'sum')
    )
)
season_finals['final_win_pct'] = season_finals['final_wins'] / season_finals['final_games']
season_finals = season_finals.sort_values(
    ['season_start', 'final_wins', 'final_point_diff'],
    ascending=[True, False, False]
)
season_finals['final_rank_proxy'] = season_finals.groupby('season_start').cumcount() + 1
season_finals['top8_proxy'] = season_finals['final_rank_proxy'].le(8)

display(season_finals.head(20))
print('Equipos-temporada:', len(season_finals))
print('Temporadas:', season_finals['season'].nunique())


In [ ]:
early_frames = []

for k in range(1, 11):
    early = (
        league_games.loc[league_games['game_number'].le(k)]
        .groupby(['season_start', 'season', 'team_code'], as_index=False)
        .agg(early_games=('win', 'size'), early_wins=('win', 'sum'))
    )
    early = early.loc[early['early_games'].eq(k)].copy()
    early['k'] = k
    early['early_win_pct'] = early['early_wins'] / k
    early_frames.append(early)

early_records = pd.concat(early_frames, ignore_index=True)
model_table = early_records.merge(
    season_finals,
    on=['season_start', 'season', 'team_code'],
    how='inner',
    validate='many_to_one'
)

signal_rows = []
for k, group in model_table.groupby('k'):
    pearson = group['early_win_pct'].corr(group['final_win_pct'], method='pearson')
    spearman = group['early_win_pct'].corr(group['final_win_pct'], method='spearman')
    baseline_mae = (group['final_win_pct'] - 0.5).abs().mean()
    early_mae = (group['final_win_pct'] - group['early_win_pct']).abs().mean()
    signal_rows.append({
        'k': k,
        'team_seasons': len(group),
        'pearson_r': pearson,
        'spearman_rho': spearman,
        'r_squared_descriptive': pearson ** 2,
        'mae_using_50_pct': baseline_mae,
        'mae_using_early_record': early_mae
    })

signal = pd.DataFrame(signal_rows)
display(signal.round(3))


R² se presenta como medida descriptiva de asociación lineal dentro de la muestra, no como precisión predictiva fuera de muestra. La comparación de errores añade contexto: si extrapolar el balance inicial no mejora el sencillo 50%, todavía sabemos poco.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.lineplot(
    data=signal,
    x='k',
    y='r_squared_descriptive',
    marker='o',
    color=AZUL_BASKONIA,
    linewidth=2.5,
    ax=axes[0]
)
axes[0].set(
    title='La clasificación gana señal con las jornadas',
    xlabel='Partidos disputados',
    ylabel='R² entre balance inicial y balance final'
)
axes[0].set_xticks(range(1, 11))

error_plot = signal.melt(
    id_vars='k',
    value_vars=['mae_using_50_pct', 'mae_using_early_record'],
    var_name='forecast',
    value_name='mae'
)
labels = {
    'mae_using_50_pct': 'Suponer 50%',
    'mae_using_early_record': 'Extrapolar el inicio'
}
error_plot['forecast'] = error_plot['forecast'].map(labels)

sns.lineplot(
    data=error_plot,
    x='k',
    y='mae',
    hue='forecast',
    marker='o',
    palette=[GRIS, ROJO_BASKONIA],
    linewidth=2.2,
    ax=axes[1]
)
axes[1].set(
    title='¿Cuándo mejora la predicción ingenua?',
    xlabel='Partidos disputados',
    ylabel='Error absoluto medio sobre el % final'
)
axes[1].set_xticks(range(1, 11))
axes[1].legend(title=None, frameon=False)

sns.despine()
plt.tight_layout()
plt.show()


### Qué ocurrió con cada tipo de salida

Se agrupan los equipos por victorias después de 1, 2, 3, 5 y 10 partidos. Las tasas con muestras pequeñas deben leerse junto al número de casos.

In [ ]:
start_outcomes = (
    model_table.loc[model_table['k'].isin([1, 2, 3, 5, 10])]
    .groupby(['k', 'early_wins'], as_index=False)
    .agg(
        cases=('team_code', 'size'),
        avg_final_win_pct=('final_win_pct', 'mean'),
        median_final_win_pct=('final_win_pct', 'median'),
        top8_proxy_rate=('top8_proxy', 'mean')
    )
)
start_outcomes['avg_final_win_pct'] *= 100
start_outcomes['median_final_win_pct'] *= 100
start_outcomes['top8_proxy_rate'] *= 100

display(start_outcomes.round(1))


### Los comienzos de Baskonia en la era de liga

In [ ]:
baskonia_starts = (
    model_table.loc[
        model_table['team_code'].eq(BASKONIA)
        & model_table['k'].isin([1, 2, 3, 5, 10])
    ]
    [[
        'season', 'k', 'early_wins', 'final_wins', 'final_games',
        'final_win_pct', 'final_rank_proxy', 'top8_proxy'
    ]]
    .sort_values(['season', 'k'])
)
baskonia_starts['final_win_pct'] = 100 * baskonia_starts['final_win_pct']

display(baskonia_starts.round({'final_win_pct': 1}))


## 3. Fotografía de 2026-27

Esta celda descarga la temporada actual, pero solo conserva partidos terminados hasta el corte editorial. Así evita que una nueva ejecución meses después reescriba silenciosamente la premisa del artículo.

In [ ]:
current_payload, current_origin = fetch_season_games(CURRENT_START_YEAR, pause_seconds=0)
current_rows = [flatten_game(game, CURRENT_START_YEAR) for game in records_from_payload(current_payload)]
current_games = pd.DataFrame(current_rows)
current_games['date'] = pd.to_datetime(current_games['date'], errors='coerce', utc=True)

current_played = current_games.loc[
    current_games['played']
    & current_games['date'].le(AS_OF_DATE)
    & current_games['local_score'].notna()
    & current_games['road_score'].notna()
].copy()

baskonia_current = current_played.loc[
    current_played['local_code'].eq(BASKONIA)
    | current_played['road_code'].eq(BASKONIA)
].sort_values(['date', 'game_code'])

def baskonia_won(row):
    if row['local_code'] == BASKONIA:
        return int(row['local_score'] > row['road_score'])
    return int(row['road_score'] > row['local_score'])

if baskonia_current.empty:
    print('No había partidos terminados de Baskonia en el corte elegido.')
else:
    current_n = len(baskonia_current)
    current_wins = int(baskonia_current.apply(baskonia_won, axis=1).sum())
    analogs = model_table.loc[
        model_table['k'].eq(current_n)
        & model_table['early_wins'].eq(current_wins)
    ].copy()

    print(f'Baskonia en el corte: {current_wins}-{current_n - current_wins}')
    print(f'Casos históricos comparables: {len(analogs)}')
    print(f'Porcentaje final medio: {100 * analogs["final_win_pct"].mean():.1f}%')
    print(f'Tasa Top 8 proxy: {100 * analogs["top8_proxy"].mean():.1f}%')
    display(baskonia_current[['date', 'local_name', 'local_score', 'road_score', 'road_name']])


## 4. Resumen editorial automático

Esta celda reúne únicamente cifras calculadas. Sirve como puente entre el análisis y el artículo y evita copiar números a mano antes de cerrar la versión final.

In [ ]:
baskonia_history = club_history.loc[club_history['team_code'].eq(BASKONIA)].iloc[0]
ranked = established_clubs.reset_index()
baskonia_rank = int(ranked.loc[ranked['team_code'].eq(BASKONIA), 'rank_wins'].iloc[0])

key_signal = signal.loc[signal['k'].isin([1, 2, 3, 5, 10])].set_index('k')

editorial_summary = pd.DataFrame([
    {'metric': 'Temporadas disponibles con Baskonia', 'value': int(baskonia_history['seasons'])},
    {'metric': 'Victorias de Baskonia', 'value': int(baskonia_history['wins'])},
    {'metric': 'Porcentaje histórico de victorias', 'value': round(baskonia_history['win_pct'], 1)},
    {'metric': 'Puesto en victorias entre clubes con 10+ temporadas', 'value': baskonia_rank},
    {'metric': 'Finales / Final Four documentadas', 'value': len(great_rounds)},
    {'metric': 'R² después de 1 partido', 'value': round(key_signal.loc[1, 'r_squared_descriptive'], 3)},
    {'metric': 'R² después de 2 partidos', 'value': round(key_signal.loc[2, 'r_squared_descriptive'], 3)},
    {'metric': 'R² después de 5 partidos', 'value': round(key_signal.loc[5, 'r_squared_descriptive'], 3)},
    {'metric': 'R² después de 10 partidos', 'value': round(key_signal.loc[10, 'r_squared_descriptive'], 3)}
])

display(editorial_summary)


In [ ]:
club_history.to_csv(PROCESSED_DIR / 'club_history.csv', index=False, encoding='utf-8-sig')
baskonia_seasons.to_csv(PROCESSED_DIR / 'baskonia_seasons.csv', index=False, encoding='utf-8-sig')
signal.to_csv(PROCESSED_DIR / 'early_start_signal.csv', index=False, encoding='utf-8-sig')
start_outcomes.to_csv(PROCESSED_DIR / 'early_start_outcomes.csv', index=False, encoding='utf-8-sig')
baskonia_starts.to_csv(PROCESSED_DIR / 'baskonia_starts.csv', index=False, encoding='utf-8-sig')
editorial_summary.to_csv(EXPORT_DIR / 'resumen_editorial.csv', index=False, encoding='utf-8-sig')
df_download_log.to_csv(EXPORT_DIR / 'registro_descargas.csv', index=False, encoding='utf-8-sig')

print('Tablas exportadas a data/processed y exports')


## Conclusiones que el artículo deberá comprobar

1. **Grandeza por acumulación:** si Baskonia aparece cerca de los primeros puestos en temporadas y victorias, la continuidad europea no es una impresión sentimental.
2. **Grandeza sin esconder el límite:** una final y cinco Final Four sostienen la profundidad histórica, pero la ausencia de título debe figurar en la tesis.
3. **Dos resultados siguen siendo una señal débil:** si el R² tras dos jornadas continúa siendo bajo, un 0-2 debe leerse como una alerta, no como una sentencia.
4. **La señal tiene un calendario:** el punto en el que extrapolar el inicio reduce el error respecto al 50% ofrece un umbral narrativo mucho más útil que el tópico de que la temporada es larga.
5. **La comparación correcta es histórica:** los análogos permiten hablar en probabilidades y no en certezas sobre la campaña actual.

Estas frases son hipótesis de redacción. Solo deben publicarse cuando las celdas ejecutadas las respalden.

## Limitaciones

- Los formatos de Euroliga han cambiado. Las victorias totales miden huella, no igualdad perfecta de oportunidades.
- El análisis de comienzos se restringe a la era de liga regular para mejorar la comparabilidad.
- Top 8 proxy ordena victorias y diferencia de puntos; no reproduce todos los criterios oficiales de desempate.
- R² es descriptivo y no implica causalidad ni garantiza predicción fuera de muestra.
- La final de 2001 se separa de las Final Four porque aquella edición terminó con una serie al mejor de cinco.
- El servicio de datos puede modificar nombres o disponibilidad. El registro de descarga y los JSON en caché forman parte de la evidencia reproducible.


## Fuentes

- EuroLeague Basketball API: https://api-live.euroleague.net/
- Perfil histórico de Baskonia: https://admin.euroleague.net/competition/teams/showteam?clubcode=bas&lang=en&p=3&seasoncode=E2001&tabid=88
- Recuentos históricos de semifinales / Final Four: https://mediacentre.euroleague.net/mediacentre/en/press_releases/5
- Documentación técnica contrastada del esquema público: https://github.com/DanielTomaro13/sportsdata-mcp/blob/main/documentation/EuroLeague.md
- Wrapper de referencia: https://github.com/giasemidis/euroleague_api
